# Yeast Dataset Preparation Tutorial

This tutorial will guide you through the process of:

1. Setting up the Yeast dataset directory structure
2. Converting Yeast dataset annotations to YOLO format
3. Creating training and validation text files

## Required Dependencies

First, let's ensure we have all necessary packages installed:

In [ ]:
!pip install torch tqdm opencv-python matplotlib numpy

## Download dataset

Details about the dataset can be found in this GitHub repository: [Yeast-in-Microstructures-Dataset](https://github.com/ChristophReich1996/Yeast-in-Microstructures-Dataset/tree/main?tab=readme-ov-file)


In [ ]:
!wget "https://tudatalib.ulb.tu-darmstadt.de/bitstream/handle/tudatalib/3799/yeast_cell_in_microstructures_dataset.zip?sequence=1&isAllowed=y" -O yeast_cell_in_microstructures_dataset.zip

## 1. Creating Directory Structure

We'll start by creating the necessary directory structure for the Yeast dataset:

```
Dataset/Yeast/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── visualizations/
    ├── train/
    ├── val/
    └── test/
```

In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def create_directory_structure():
    base_dir = "Dataset/Yeast"
    dirs = [
        "images/train",
        "images/val",
        "images/test",
        "labels/train",
        "labels/val",
        "labels/test",
        "visualizations/train",
        "visualizations/val", 
        "visualizations/test"
    ]
    
    for dir_path in dirs:
        Path(f"{base_dir}/{dir_path}").mkdir(parents=True, exist_ok=True)
    
    return base_dir

base_dir = create_directory_structure()
print(f"Created directory structure in {base_dir}")

## 2. Converting Yeast Dataset to YOLO Format

Next, we'll convert the Yeast dataset format to YOLO format. The YOLO format is:

```
[class_id] [x_center] [y_center] [width] [height]
```

where:
- class_id: 0 for trap, 1 for cell
- x_center, y_center: center coordinates normalized to [0, 1]
- width, height: bbox dimensions normalized to [0, 1]

We'll also convert the PyTorch tensor images to JPG format and create visualizations.

In [ ]:
def convert_to_yolo_format_and_save_images():
    # Class names (cell = 1, trap = 0)
    class_names = ["trap", "cell"]

    def convert_bbox_to_yolo_format(size, box):
        dw = 1. / size[0]
        dh = 1. / size[1]
        
        x0, y0, x1, y1 = box
        x_center = (x0 + x1) / 2 * dw
        y_center = (y0 + y1) / 2 * dh
        width = (x1 - x0) * dw
        height = (y1 - y0) * dh
        
        return (x_center, y_center, width, height)

    def process_annotations():
        # Process each image for the dataset (train, val, test)
        for dataset in ['train', 'val', 'test']:
            print(f"\nProcessing {dataset}...")
            
            # Load annotation files
            bbox_dir = f'Dataset/yeast_cell_in_microstructures_dataset/{dataset}/bounding_boxes'
            class_dir = f'Dataset/yeast_cell_in_microstructures_dataset/{dataset}/classes'
            input_dir = f'Dataset/yeast_cell_in_microstructures_dataset/{dataset}/inputs'
            
            image_files = [f for f in os.listdir(input_dir) if f.endswith('.pt')]
            
            for img_file in tqdm(image_files):
                # Load image
                image_path = os.path.join(input_dir, img_file)
                image = torch.load(image_path, weights_only=True)  # Tensor of shape [1, 128, 128]
                image = image.squeeze(0).numpy()  # Remove the channel dimension
                
                # Normalize and save image in JPG format
                image_normalized = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
                output_image_path = os.path.join(f"Dataset/Yeast/images/{dataset}", os.path.splitext(img_file)[0] + '.jpg')
                cv2.imwrite(output_image_path, cv2.cvtColor(image_normalized, cv2.COLOR_GRAY2BGR))
                
                # Get bounding box, classes
                bbox_file = os.path.join(bbox_dir, img_file)
                class_file = os.path.join(class_dir, img_file)

                bounding_boxes = torch.load(bbox_file, weights_only=True)  # [N, 4 (x0, y0, x1, y1)]
                classes = torch.load(class_file, weights_only=True)  # [N] (each class is 0 for trap, 1 for cell)

                # Convert bounding boxes to YOLO format
                height, width = image.shape
                yolo_labels = []

                for i in range(len(bounding_boxes)):
                    box = bounding_boxes[i].numpy()
                    class_id = int(classes[i].numpy())
                    yolo_bbox = convert_bbox_to_yolo_format((width, height), box)
                    yolo_labels.append(f"{class_id} " + " ".join([f"{x:.6f}" for x in yolo_bbox]))
                
                # Save YOLO format label
                label_dir = f"Dataset/Yeast/labels/{dataset}"
                os.makedirs(label_dir, exist_ok=True)
                label_file = os.path.join(label_dir, os.path.splitext(img_file)[0] + '.txt')

                with open(label_file, 'w') as f:
                    f.write("\n".join(yolo_labels))

                # Create visualization
                visualize_image_with_bboxes(image, bounding_boxes, classes, dataset, img_file)

    def visualize_image_with_bboxes(image, bounding_boxes, classes, dataset, img_file):
        """Visualize the image with bounding boxes and class labels."""
        plt.figure(figsize=(8, 8))
        plt.imshow(image, cmap='gray')
        
        # Plot bounding boxes
        for i, box in enumerate(bounding_boxes):
            class_id = int(classes[i].numpy())
            x0, y0, x1, y1 = box.numpy()
            color = 'green' if class_id == 1 else 'red'  # Green for cell, Red for trap
            
            # Draw the bounding box
            rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor=color, facecolor='none')
            plt.gca().add_patch(rect)
            
            # Add class label
            class_name = class_names[class_id]
            plt.text(x0, y0, class_name, fontsize=12, color=color, bbox=dict(facecolor='white', alpha=0.7))
        
        plt.axis('off')
        plt.tight_layout()
        
        # Save visualization
        vis_dir = f"Dataset/Yeast/visualizations/{dataset}"
        os.makedirs(vis_dir, exist_ok=True)
        plt.savefig(os.path.join(vis_dir, os.path.splitext(img_file)[0] + '_vis.jpg'))
        plt.close()

    # Run processing and visualization
    process_annotations()

# Run YOLO format conversion and image saving
convert_to_yolo_format_and_save_images()

## 3. Creating Training, Validation, and Test Text Files

Finally, we'll create text files listing all images in the training, validation and test sets:

In [ ]:
def create_image_lists():
    dataset_path = Path('./Dataset/Yeast')
    
    # Create train.txt
    train_images = list((dataset_path / 'images' / 'train').glob('*.jpg'))
    with open(dataset_path / 'train.txt', 'w') as f:
        for img_path in train_images:
            f.write(f'./Dataset/Yeast/images/train/{img_path.name}\n')
    
    # Create val.txt
    val_images = list((dataset_path / 'images' / 'val').glob('*.jpg'))
    with open(dataset_path / 'val.txt', 'w') as f:
        for img_path in val_images:
            f.write(f'./Dataset/Yeast/images/val/{img_path.name}\n')
    
    # Create test.txt
    test_images = list((dataset_path / 'images' / 'test').glob('*.jpg'))
    with open(dataset_path / 'test.txt', 'w') as f:
        for img_path in test_images:
            f.write(f'./Dataset/Yeast/images/test/{img_path.name}\n')

    print(f"Created train.txt with {len(train_images)} images")
    print(f"Created val.txt with {len(val_images)} images")
    print(f"Created test.txt with {len(test_images)} images")

create_image_lists()

## Verification

Let's verify that our dataset is properly structured and all necessary files are in place:

In [ ]:
def verify_dataset():
    base_dir = Path("Dataset/Yeast")
    
    # Check directory structure
    required_dirs = [
        "images/train",
        "images/val",
        "images/test",
        "labels/train",
        "labels/val",
        "labels/test",
        "visualizations/train",
        "visualizations/val",
        "visualizations/test"
    ]
    
    for dir_path in required_dirs:
        full_path = base_dir / dir_path
        if not full_path.exists():
            print(f"❌ Missing directory: {dir_path}")
        else:
            print(f"✅ Found directory: {dir_path}")
    
    # Check text files
    for txt_file in ["train.txt", "val.txt", "test.txt"]:
        if (base_dir / txt_file).exists():
            print(f"✅ Found file: {txt_file}")
        else:
            print(f"❌ Missing file: {txt_file}")
    
    # Check label files
    print("\nChecking label files format...")
    for dataset in ["train", "val", "test"]:
        label_dir = base_dir / "labels" / dataset
        if label_dir.exists():
            label_files = list(label_dir.glob("*.txt"))
            if label_files:
                sample_file = label_files[0]
                print(f"\nSample {dataset} label file ({sample_file.name}):")
                with open(sample_file) as f:
                    print(f.read().strip())

verify_dataset()

## Visualization of Sample Images with Bounding Boxes

Let's display some sample images with their bounding boxes from the training and validation sets:

In [ ]:
def display_sample_images(dataset='train', num_images=9):
    base_dir = Path("Dataset/Yeast")
    vis_dir = base_dir / "visualizations" / dataset
    
    # Get all visualization images
    vis_images = list(vis_dir.glob("*_vis.jpg"))
    if not vis_images:
        print(f"No visualization images found in {vis_dir}")
        return
    
    # Select random images
    import random
    selected_images = random.sample(vis_images, min(num_images, len(vis_images)))
    
    # Create a grid of images
    rows = int(np.ceil(np.sqrt(len(selected_images))))
    cols = rows
    fig, axes = plt.subplots(rows, cols, figsize=(15, 15))
    axes = axes.ravel()
    
    for idx, img_path in enumerate(selected_images):
        img = plt.imread(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"{dataset}: {img_path.stem}")
    
    # Hide empty subplots
    for idx in range(len(selected_images), rows*cols):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Training Images with Bounding Boxes:")
display_sample_images('train')

print("\nValidation Images with Bounding Boxes:")
display_sample_images('val')

## Conclusion

You have now successfully:

1. Created the necessary directory structure for the Yeast dataset
2. Converted PyTorch tensor images to JPG format
3. Converted bounding box annotations to YOLO format
4. Created visualizations of the dataset
5. Created text files listing all training, validation, and test images
6. Verified the dataset structure and format
7. Visualized sample images with bounding boxes

The dataset is now ready to be used for training YOLO models. The directory structure looks like this:

```
Dataset/Yeast/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── visualizations/
    ├── train/
    ├── val/
    └── test/
```
